In [1]:
# imports and config
import numpy as np
import pandas as pd
from pathlib import Path
import scipy.stats
from tqdm import tqdm
from plotnine import *
rng = np.random.default_rng()

In [2]:
dir = Path('account_example_n20000')


In [3]:

config = pd.read_csv(dir / 'config.csv')
scan_perf = pd.read_csv(dir / 'scan_perf.csv')


In [4]:
study_rate = float(config.loc[0, 'study_rate'])
incoming_int_columns = (
    'presumed_indications',
    'n_true_positive', 'n_false_positive',
    'n_false_negative', 'n_true_negative',
    'n compromised | no intervention',
    'n compromised | intervention',
    'n at risk | intervention',
    'n at risk | no intervention',
    'n_at_risk',
)
for v in incoming_int_columns:
    scan_perf[v] = scan_perf[v].astype(int)

In [5]:
# shared formatting
value_formats = {
    'n_true_positive': '{:0.1f}',
    'n_false_positive': '{:0.1f}',
    'n_false_negative': '{:0.1f}',
    'n_true_negative': '{:0.1f}',
    'precision': '{:.1%}',
    'recall': '{:.1%}',
    'E[indications]': '{:0.1f}',
    'sd[indications]': '{:0.1f}', 
    'cost with intervention': "${:,.2f}", 
    'scan value': "${:,.2f}",
    'scan_cost': "${:,.2f}",
    'intervention_cost': "${:,.2f}",
    'intervention_effectiveness': '{:.0%}',
    'expected_loss': "${:,.2f}",
    'E[compromises | no action]': '{:0.1f}', 
    'E[compromises | intervened]': '{:0.1f}',
    'E[at risk | no action]': '{:0.1f}', 
    'E[at risk | intervened]': '{:0.1f}',
    'intervention cost': "${:,.2f}", 
    'post intervention losses': "${:,.2f}", 
    'cost with intervention': "${:,.2f}",
    'no intervention losses': "${:,.2f}",
    'scan cost': "${:,.2f}",
    'return on spend': '{:.0%}', 
    'at risk rate outside selection': '{:.3%}', 
    'attacker_coverage': '{:.1%}',
    'presumed_at_risk_rate': "{:0.2%}",
    'calibration_possible_positives': '{:.0f}', 
    'calibration_true_positives': '{:.0f}',
    'calibration_possible_negatives': '{:.0f}', 
    'calibration_true_negatives': '{:.0f}',
    'n_accounts': '{:.0f}', 
    'study_rate': '{:0.2%}',
    'anticipated at risk': '{:0.1f}',
    'anticipated attacks': '{:0.1f}',
    'n compromised | no intervention': '{:0.1f}',
    'n compromised | intervention': '{:0.1f}',
    'n_at_risk': '{:0.1f}',
    'n_at_risk | no intervention': '{:0.1f}',
    'n at risk | intervention': '{:0.1f}',
    'attacker_coverage': '{:0.1%}',
    'last_year_loss': "${:,.2f}",
    'sd_loss': "${:,.2f}",
    "n at risk | no intervention": '{:0.1f}',
    'modeled_sensitivity': '{:0.1%}',
    'modeled_specificity': '{:0.1%}',
}
index_formats = lambda x: f"{x:0.2%}" if isinstance(x, float) else str(x)

In [6]:
# get the matching rates
group_vars = ['presumed_at_risk_rate', 'presumed_indications']
rate_frame = scan_perf[group_vars + ['E[indications]']].groupby(group_vars).mean()
rate_frame = rate_frame.reset_index(drop=False, inplace=False)
rate_frame['diff'] = np.abs(rate_frame['presumed_indications'] - rate_frame['E[indications]'])
rate_frame['min'] = rate_frame.groupby(['presumed_at_risk_rate']).transform('min')['diff']
rate_frame = rate_frame.loc[rate_frame['diff'] <= rate_frame['min'], :]
rate_frame = rate_frame[group_vars].groupby(group_vars).first().reset_index(drop=False, inplace=False)
rate_frame.style.format(value_formats).hide(axis="index")

presumed_at_risk_rate,presumed_indications
1.00%,1300
1.50%,1400
2.00%,1400
2.50%,1500
3.00%,1600


In [7]:
ind_frame = (
    scan_perf
        .groupby(group_vars)
        [['E[indications]', 'modeled_sensitivity', 'modeled_specificity']]
        .mean()
)
(
    ind_frame
        .style
        .format_index(index_formats)
        .format(value_formats)
)

In [8]:
target_indications_value = int(
    rate_frame.loc[rate_frame['presumed_at_risk_rate'] == study_rate, 'presumed_indications'].values[0])

target_indications_value

1400

In [9]:
relevant_examples = scan_perf.loc[
    (scan_perf['presumed_at_risk_rate'] == study_rate)
    & (scan_perf['presumed_indications'] == target_indications_value),
    :].reset_index(drop=True, inplace=False)

(
    relevant_examples
        .head()
        .style
        .format({k: v for k, v in value_formats.items() if k not in incoming_int_columns}).hide(axis="index")
)

modeled_sensitivity,modeled_specificity,n_at_risk,n_safe,presumed_at_risk_rate,presumed_indications,n_true_positive,n_false_positive,n_false_negative,n_true_negative,attacker_coverage,n at risk | intervention,n at risk | no intervention,n compromised | no intervention,n compromised | intervention,no intervention losses,post intervention losses,E[indications],var[indications]
62.2%,94.2%,402,19598,2.00%,1400,254,1146,148,18452,12.8%,199,402,48,21,"$630,665.44","$237,952.38",1430.8,1362.337278
86.9%,94.8%,441,19559,2.00%,1400,374,1026,67,18533,13.7%,137,441,64,23,"$802,939.90","$294,656.38",1430.8,1362.337278
65.9%,94.0%,405,19595,2.00%,1400,257,1143,148,18452,10.0%,196,405,42,17,"$503,754.27","$209,724.51",1430.8,1362.337278
79.7%,94.5%,407,19593,2.00%,1400,331,1069,76,18524,9.9%,130,407,52,19,"$566,125.16","$227,331.84",1430.8,1362.337278
67.0%,94.2%,411,19589,2.00%,1400,272,1128,139,18461,15.9%,197,411,73,36,"$887,520.55","$409,437.90",1430.8,1362.337278


In [10]:
relevant_examples = relevant_examples[['n_true_positive', 'n_false_positive', 'n_false_negative', 'n_true_negative']].copy()
relevant_examples.rename(columns={
    'n_true_positive': 'at_risk_found', 
    'n_false_positive': 'safe_mistaken', 
    'n_false_negative': 'at_risk_missed', 
    'n_true_negative': 'safe_left_alone',
}, inplace=True)


In [11]:
html_map = {
    "at_risk_found": "at_risk_<sub>found</sub>",
    'safe_mistaken': 'safe_<sub>mistaken</sub>',
    'at_risk_missed': 'at_risk_<sub>missed<sub>',
    'safe_left_alone': 'safe_left_<sub>alone</sub>',
}
(
    relevant_examples
        .head(100)
        .style.format_index(
            lambda name: html_map[name] if name in html_map else name,
            axis="columns",
        )
        .hide(axis="index")
)

at_risk_found,safe_mistaken,at_risk_missed,safe_left_alone
254,1146,148,18452
374,1026,67,18533
257,1143,148,18452
331,1069,76,18524
272,1128,139,18461
361,1039,28,18572
307,1093,102,18498
383,1017,31,18569
324,1076,76,18524
286,1114,78,18522


In [ ]:
coin_frame = pd.DataFrame({
    'is_win': rng.binomial(n=1, p=1/6, size=1000),
})


,is_win
0,0
1,0
2,1
3,0
4,1
...,...
995,0
996,0
997,1
998,1


In [14]:
(
    coin_frame
        .head(100)
        .style
        .hide(axis="index")
)

is_win
0
0
1
0
1
0
1
0
1
0
